In [1]:
pip install datasets transformers evaluate scikit-learn torch


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "TRUE"

In [11]:
from datasets import load_dataset

dataset = load_dataset("glue", "qqp")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}


In [13]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["question1"],
        example["question2"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 390965/390965 [00:22<00:00, 17668.44 examples/s]


In [15]:
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
small_eval = tokenized_dataset["validation"].shuffle(seed=42).select(range(1000))

In [16]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"]
    }

training_args = TrainingArguments(
    output_dir="./bert-qqp-results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




















































































































































































































































































































































































































































                                                         


                                      

                                           
  2%|▏         | 1098/68223 [66:24:26<10:18:41,  1.81it/s]






Attempted to log scalar metric eval_loss:
0.4254181981086731
Attempted to log scalar metric eval_accuracy:
0.793
Attempted to log scalar metric eval_f1:
0.7369758576874206
Attempted to log scalar metric eval_runtime:
11.5689
Attempted to log scalar metric eval_samples_per_second:
86.438
Attempted to log scalar metric eval_steps_per_second:
5.446
Attempted to log scalar metric epoch:
1.0
{'eval_loss': 0.4254181981086731, 'eval_accuracy': 0.793, 'eval_f1': 0.7369758576874206, 'eval_runtime': 11.5689, 'eval_samples_per_second': 86.438, 'eval_steps_per_second': 5.446, 'epoch': 1.0}


                                                          

                                           
100%|██████████| 313/313 [17:49<00:00,  3.42s/it]

Attempted to log scalar metric train_runtime:
1069.3509
Attempted to log scalar metric train_samples_per_second:
4.676
Attempted to log scalar metric train_steps_per_second:
0.293
Attempted to log scalar metric total_flos:
328888819200000.0
Attempted to log scalar metric train_loss:
0.5020406802241414
Attempted to log scalar metric epoch:
1.0
{'train_runtime': 1069.3509, 'train_samples_per_second': 4.676, 'train_steps_per_second': 0.293, 'train_loss': 0.5020406802241414, 'epoch': 1.0}


TrainOutput(global_step=313, training_loss=0.5020406802241414, metrics={'train_runtime': 1069.3509, 'train_samples_per_second': 4.676, 'train_steps_per_second': 0.293, 'total_flos': 328888819200000.0, 'train_loss': 0.5020406802241414, 'epoch': 1.0})

In [17]:
trainer.evaluate()































































100%|██████████| 63/63 [00:09<00:00,  6.40it/s]

Attempted to log scalar metric eval_loss:
0.4254181981086731
Attempted to log scalar metric eval_accuracy:
0.793
Attempted to log scalar metric eval_f1:
0.7369758576874206
Attempted to log scalar metric eval_runtime:
19.0185
Attempted to log scalar metric eval_samples_per_second:
52.58
Attempted to log scalar metric eval_steps_per_second:
3.313
Attempted to log scalar metric epoch:
1.0


{'eval_loss': 0.4254181981086731,
 'eval_accuracy': 0.793,
 'eval_f1': 0.7369758576874206,
 'eval_runtime': 19.0185,
 'eval_samples_per_second': 52.58,
 'eval_steps_per_second': 3.313,
 'epoch': 1.0}